In [ ]:
import torch
from botorch.exceptions import InputDataWarning

from bott.problem import OptimizationProblem
from bott.physics_models import simulate_cbed
from bott.optimization import run_one_trial

import warnings
warnings.filterwarnings("ignore", category=InputDataWarning) 
# InputDataWarning: Data (outcome observations) is not standardized (std = tensor([5.3673e-11, 5.0607e-10, 4.8782e-10, 7.7036e-11, 5.1824e-14],
#        dtype=torch.float64), mean = tensor([0.0000e+00, 0.0000e+00, 8.4703e-22, 0.0000e+00, 0.0000e+00],
#        dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
#   check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)

test 2

In [ ]:
break

In [ ]:
break

In [ ]:
ground_truth = torch.Tensor(simulate_cbed(20,10,-10, device_simu='cpu')) # abtem takes "cpu" or "gpu"

In [ ]:
# OptimizationProblem would keep all the tensor on the specified device
problem = OptimizationProblem(ground_truth=ground_truth,
                              output_path='./output', 
                              save_results=True, 
                              reduction_params={'reduction_type':'square', 'reduction_kwargs':{'num_tiles':2}},
                              loss_params={'loss_type':'SSE', 'dp_pow': 0.5}, 
                              norm_arr=False,
                              dim=3, 
                              bounds=[(15,25), (-20, 20), (-20, 20)],
                              noise_std=0,
                              dtype=torch.float64, 
                              device='cpu'
                              ) # "cpu" or "cuda" for physics simulation

In [ ]:
loss_func = problem.loss_func
reduction_true = problem.reduction_true.to(torch.double)
measurement_true = problem.measurement_true.to(torch.double)

In [ ]:
from botorch.acquisition.objective import GenericMCObjective, MCAcquisitionObjective

In [ ]:
objective = GenericMCObjective(lambda Y, X=None: -1*(loss_func(Y[...,:-1], reduction_true,reduce=True) + Y[...,-1]))

# Testing

In [ ]:
trial=30

In [ ]:
import numpy as np
import random
from botorch.utils.sampling import draw_sobol_samples

torch.manual_seed(trial)
np.random.seed(trial)
random.seed(trial)

# random initial points and calculate intermediate outputs [cbed]
X=draw_sobol_samples(bounds=problem.bounds.to(torch.double),n=2,q=1).squeeze(-2) # X = [n_init, problem.dim], note that X is by default on cpu because problem.bounds is also on cpu

# Physical images output 
input_params = X.tolist()
outputs_np = np.array([problem.get_physics_simu(*param) for param in input_params])
image_output = torch.tensor(outputs_np, dtype=torch.double, device='cpu')
print(f"image output shape {image_output.shape}")

In [ ]:
image_output = torch.cat((image_output,problem.measurement_true.unsqueeze(0)),dim=0)

In [ ]:
pixelLoss  = loss_func(y_simu=image_output, y_true=measurement_true, reduce=False).unsqueeze(-1) # pixelLoss = [n_init, 1]


In [ ]:
pixelLoss # The last one should return 0

In [ ]:
y_reduction = problem.reduction_func(image_output).to(torch.double)  # [n_init, num_tiles]
reductionLoss = loss_func(y_simu=y_reduction, y_true=reduction_true, reduce=False).unsqueeze(-1) # [n_init, 1]

In [ ]:
y_reduction, reduction_true # the last one of reduction should be the same as the reduction true

In [ ]:
epsilon = pixelLoss - reductionLoss
y_value = torch.cat((y_reduction,epsilon),dim=-1)

In [ ]:
y_value_batch = y_value.unsqueeze(0).unsqueeze(0).repeat(5,4,1,1)

In [ ]:
y_value_batch.shape

In [ ]:
objective(y_value_batch).shape

In [ ]:
break

# Run one trial

In [ ]:
sbatch -J EM_test -o ../output/EMtest_%j.out -e ../output/EMtest%j.err --requeue submit.sub
/home/pb482/bott/output

In [ ]:
run_one_trial(problem_name='EICF', 
              problem=problem, 
              algo='EICF', 
              trial=5, 
              n_init_evals=2, 
              max_iter=50, 
              objective=None,
              dtype=torch.float64,
              device_botorch='cpu'
              )

In [ ]:
break

In [ ]:
image_test = torch.rand(2,315,315).to(torch.double)

In [ ]:
image_test= torch.cat((image_test,problem.measurement_true.unsqueeze(0)),dim=0)

In [ ]:
pixelLoss  = problem.loss_func(y_simu=image_test, y_true=problem.measurement_true, reduce=False).unsqueeze(-1) # pixelLoss = [n_init, 1]


In [ ]:
y_reduction = problem.reduction_func(image_test)

In [ ]:
y_reduction

In [ ]:
problem.reduction_true

In [ ]:
reductionLoss = problem.loss_func(y_simu=y_reduction, y_true=problem.reduction_true, reduce=False).unsqueeze(-1) # [n_init, 1]


In [ ]:
reductionLoss

In [ ]:
pixelLoss-reductionLoss

In [ ]:
pixelLoss